In [1]:
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.metrics.pairwise import rbf_kernel

In [2]:
class NRLMF:
    def __init__(self, c=5, gamma=1, lambda_d=1, lambda_t=1, r=10, alpha=0.01, beta=0.01, theta=0.01, max_iter=100):

        self.lambda_d = lambda_d
        self.lambda_t = lambda_t
        self.r = r
        self.alpha = alpha
        self.beta = beta
        self.max_iter = max_iter

    def fix_model(self, Y, Sd, St):
        self.Y = Y
        self.Sd = Sd
        self.St = St

        self.num_drugs, self.num_targets = Y.shape

        self.D = np.random.rand(self.num_drugs, self.r)
        self.T = np.random.rand(self.num_targets, self.r)

        for _ in range(self.max_iter):
            self.update_D()
            self.update_T()

    def update_D(self):
        TT = self.T.T @ self.T
        I = np.eye(self.r)

        for i in range(self.num_drugs):
            sim_term = self.alpha * np.sum(
                [self.Sd[i, k] * self.D[k] for k in range(self.num_drugs)],
                axis=0
            )

            A = TT + self.lambda_d * I
            b = self.Y[i] @ self.T + sim_term

            self.D[i] = np.linalg.solve(A, b)

    def update_T(self):
        DD = self.D.T @ self.D
        I = np.eye(self.r)

        for j in range(self.num_targets):
            sim_term = self.beta * np.sum(
                [self.St[j, k] * self.T[k] for k in range(self.num_targets)],
                axis=0
            )

            A = DD + self.lambda_t * I
            b = self.Y[:, j] @ self.D + sim_term

            self.T[j] = np.linalg.solve(A, b)

    def predict(self):
        return self.D @ self.T.T

In [4]:
protein_data = pd.read_csv('ic_simmat_dg.txt', sep ="\t", header=0)
protein_data = protein_data.rename(columns={'Unnamed: 0': 'UniProt ID'})
protein_data = protein_data.set_index('UniProt ID')
protein_data

,hsa10008,hsa10060,hsa10369,hsa1080,hsa11254,hsa11280,hsa1134,hsa1135,hsa1136,hsa1137,...,hsa9127,hsa9132,hsa9177,hsa9254,hsa93107,hsa9311,hsa9312,hsa93589,hsa9424,hsa9992
UniProt ID,,,,,,,,,,,,,,,,,,,,,
hsa10008,1.000000,0.015248,0.034161,0.019936,0.028227,0.018513,0.029744,0.023408,0.024370,0.027928,...,0.029220,0.029711,0.031082,0.024279,0.030289,0.023927,0.020612,0.019705,0.032935,0.133162
hsa10060,0.015248,1.000000,0.016270,0.152869,0.010799,0.006184,0.012787,0.009883,0.008746,0.008635,...,0.010325,0.013282,0.010668,0.006660,0.012427,0.012296,0.012395,0.007753,0.013852,0.026888
hsa10369,0.034161,0.016270,1.000000,0.014459,0.023931,0.011132,0.020863,0.024122,0.018366,0.016886,...,0.015836,0.021055,0.019296,0.012583,0.031339,0.015766,0.017316,0.013705,0.020184,0.035570
hsa1080,0.019936,0.152869,0.014459,1.000000,0.012150,0.009885,0.012927,0.011824,0.009847,0.011184,...,0.014488,0.010177,0.012636,0.007871,0.014703,0.008741,0.008648,0.007833,0.012175,0.015236
hsa11254,0.028227,0.010799,0.023931,0.012150,1.000000,0.010857,0.020684,0.012197,0.015549,0.012362,...,0.016424,0.012572,0.013341,0.011429,0.014806,0.014121,0.011199,0.009648,0.017278,0.026196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
hsa9311,0.023927,0.012296,0.015766,0.008741,0.014121,0.010215,0.014576,0.020406,0.019828,0.023020,...,0.021243,0.016152,0.016101,0.009882,0.018946,1.000000,0.014774,0.010922,0.021427,0.024543
hsa9312,0.020612,0.012395,0.017316,0.008648,0.011199,0.019904,0.013680,0.011451,0.015233,0.014432,...,0.010727,0.068585,0.011129,0.007993,0.163978,0.014774,1.000000,0.009540,0.020947,0.018674
hsa93589,0.019705,0.007753,0.013705,0.007833,0.009648,0.006895,0.012206,0.010785,0.013395,0.008526,...,0.012068,0.009389,0.010986,0.208633,0.016470,0.010922,0.009540,1.000000,0.011365,0.016662


In [6]:
molecule_data  = pd.read_csv('ic_simmat_dc.txt', sep ="\t", header=0)
molecule_data = molecule_data.rename(columns={'Unnamed: 0': 'Drug'})
molecule_data = molecule_data.set_index('Drug')
molecule_data

,D00035,D00110,D00136,D00195,D00219,D00225,D00227,D00228,D00234,D00252,...,D04985,D04999,D05024,D05077,D05156,D05453,D05458,D05461,D06106,D06172
Drug,,,,,,,,,,,,,,,,,,,,,
D00035,1.000000,0.111111,0.062500,0.107143,0.153846,0.034483,0.000000,0.111111,0.076923,0.000000,...,0.038462,0.190476,0.069767,0.055556,0.166667,0.125000,0.000000,0.082192,0.041667,0.000000
D00110,0.111111,1.000000,0.200000,0.285714,0.294118,0.222222,0.106383,0.222222,0.217391,0.176471,...,0.078947,0.560000,0.250000,0.300000,0.191489,0.322581,0.320000,0.094118,0.054054,0.200000
D00136,0.062500,0.200000,1.000000,0.225000,0.333333,0.333333,0.142857,0.454545,0.463415,0.294118,...,0.071429,0.228571,0.306122,0.435897,0.153846,0.323529,0.156250,0.197531,0.075000,0.176471
D00195,0.107143,0.285714,0.225000,1.000000,0.216216,0.184211,0.127660,0.250000,0.212766,0.171429,...,0.105263,0.250000,0.220000,0.204545,0.187500,0.200000,0.172414,0.119048,0.081081,0.193548
D00219,0.153846,0.294118,0.333333,0.216216,1.000000,0.189189,0.083333,0.257143,0.302326,0.176471,...,0.025000,0.392857,0.153846,0.300000,0.166667,0.322581,0.222222,0.107143,0.026316,0.285714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D05453,0.125000,0.322581,0.323529,0.200000,0.322581,0.242424,0.065217,0.366667,0.261905,0.193548,...,0.027027,0.384615,0.163265,0.256410,0.204545,1.000000,0.250000,0.168831,0.028571,0.137931
D05458,0.000000,0.320000,0.156250,0.172414,0.222222,0.269231,0.171429,0.178571,0.184211,0.260870,...,0.111111,0.333333,0.166667,0.171429,0.153846,0.250000,1.000000,0.078947,0.120000,0.388889
D05461,0.082192,0.094118,0.197531,0.119048,0.107143,0.148148,0.188235,0.192308,0.220930,0.098765,...,0.111111,0.100000,0.172043,0.202381,0.193182,0.168831,0.078947,0.945205,0.100000,0.075949


In [8]:
interaction_data  = pd.read_csv('ic_admat_dgc.txt', sep ="\t", header=0)
interaction_data = interaction_data.rename(columns={'Unnamed: 0': 'UniProt ID'})
interaction_data = interaction_data.set_index('UniProt ID')
interaction_data

,D00035,D00110,D00136,D00195,D00219,D00225,D00227,D00228,D00234,D00252,...,D04985,D04999,D05024,D05077,D05156,D05453,D05458,D05461,D06106,D06172
UniProt ID,,,,,,,,,,,,,,,,,,,,,
hsa10008,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hsa10060,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hsa10369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hsa1080,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hsa11254,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
hsa9311,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
hsa9312,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
hsa93589,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
def clean_ids(df):
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip()

    return df

In [11]:
protein_data = clean_ids(protein_data)
molecule_data = clean_ids(molecule_data)
interaction_data = clean_ids(interaction_data)

In [12]:
Y = interaction_data.values.T

molecule_data = molecule_data.loc[interaction_data.columns, interaction_data.columns]
protein_data = protein_data.loc[interaction_data.index, interaction_data.index]

Sd = molecule_data.values
St = protein_data.values

Sd = Sd / (Sd.max() + 1e-10)
St = St / (St.max() + 1e-10)

protein_ids = interaction_data.index.values
molecule_ids = interaction_data.columns.values

print("Y shape:", Y.shape)
print("Sd shape:", Sd.shape)
print("St shape:", St.shape)

Y shape: (210, 204)
Sd shape: (210, 210)
St shape: (204, 204)


In [13]:
params = {
    'c': 5,
    'gamma': 1,
    'lambda_d': 1,
    'lambda_t': 1,
    'r': 10,
    'alpha': 0.01,
    'beta': 0.01,
    'theta': 0.01,
    'max_iter': 100
}

nrlmf_model = NRLMF(**params)
nrlmf_model.fix_model(Y, Sd, St)

Y_pred = nrlmf_model.predict()

print("Model training is complete")

Model training is complete


In [14]:
low_prob_pairs = []

num_positive = int(np.sum(Y))
neg_candidates = []

for i, drug in enumerate(molecule_ids):
    for j, prot in enumerate(protein_ids):
        if Y[i, j] == 0:
            neg_candidates.append((drug, prot, Y_pred[i, j]))

neg_df = pd.DataFrame(neg_candidates, columns=['Drug', 'UniProt ID', 'score'])
neg_df = neg_df.sort_values(by='score', ascending=True)

#1:1
low_prob_pairs_df = neg_df.head(num_positive).reset_index(drop=True)

In [15]:
low_prob_pairs_df

,Drug,UniProt ID,score
0,D00542,hsa1080,-0.303207
1,D00649,hsa6334,-0.218490
2,D00649,hsa6326,-0.218482
3,D00649,hsa6335,-0.218464
4,D03878,hsa2561,-0.212957
...,...,...,...
1471,D00809,hsa2564,-0.020159
1472,D00809,hsa2566,-0.020155
1473,D05461,hsa3743,-0.020102
1474,D02088,hsa3782,-0.020082


In [18]:
low_prob_pairs_df.to_csv("hard_negative_nrlmf_DTI_IC.csv", index=False)

In [16]:
print("Jumlah positive:", np.sum(Y))
print("Jumlah negative:", len(low_prob_pairs_df))

Jumlah positive: 1476
Jumlah negative: 1476


In [17]:
overlap = sum(
    interaction_data.loc[row['UniProt ID'], row['Drug']] == 1
    for _, row in low_prob_pairs_df.iterrows()
)

print("Overlap:", overlap)

Overlap: 0
